In [1]:
import pandas as pd
import itertools
import re

In [2]:
training_set = pd.read_csv('/home/s6moakba/InstructABSA/CV_validation/res_14_ate/wong_cv.csv')

In [3]:
# Remove specified characters from 'labels' and 'prediction' columns
training_set['labels'] = training_set['labels'].str.replace(r'["\[\]]', '', regex=True)
training_set['prediction'] = training_set['prediction'].str.replace(r'["\[\]]', '', regex=True)

In [4]:
training_set['labels'] = training_set['labels'].apply(lambda x: x.split(','))
training_set['prediction'] = training_set['prediction'].apply(lambda x: x.split(','))

In [5]:
training_set['aspectTerms'] = training_set['aspectTerms'].apply(eval)

In [6]:
def extract_terms(aspect_terms_list):
    terms = []
    for aspect in aspect_terms_list:
        terms.append(aspect['term'])
    return terms

training_set['terms'] = training_set['aspectTerms'].apply(extract_terms)

In [7]:
training_set = training_set[training_set.apply(lambda row: sorted(row['terms']) != sorted(row['prediction']), axis=1)]
training_set

,sentenceId,raw_text,aspectTerms,aspectCategories,labels,text,prediction,terms
0,2227,My suggestion is to eat family style because y...,"[{'term': 'dishes', 'polarity': 'neutral'}, {'...","[{'category': 'food', 'polarity': 'neutral'}]","['dishes', 'eat family style']",Definition: The output will be the aspects (bo...,['dishes'],"[dishes, eat family style]"
1,2962,Three courses - choices include excellent muss...,"[{'term': 'mussels', 'polarity': 'positive'}, ...","[{'category': 'food', 'polarity': 'positive'}]","['mussels', 'puff pastry goat cheese', 'sala...",Definition: The output will be the aspects (bo...,"['mussels', 'puff pastry goat cheese', 'sala...","[mussels, puff pastry goat cheese, salad with ..."
2,2911,"Great food, good size menu, great service and ...","[{'term': 'food', 'polarity': 'positive'}, {'t...","[{'category': 'food', 'polarity': 'positive'},...","['food', 'menu', 'service', 'setting']",Definition: The output will be the aspects (bo...,"['food', 'size menu', 'service', 'setting']","[food, menu, service, setting]"
3,672,"Dip the ingredients in with your chopsticks, s...","[{'term': 'ingredients', 'polarity': 'neutral'...","[{'category': 'food', 'polarity': 'neutral'}]","['ingredients', 'chopsticks']",Definition: The output will be the aspects (bo...,['ingredients'],"[ingredients, chopsticks]"
4,567,Food was average and creme brulee was awful - ...,"[{'term': 'Food', 'polarity': 'neutral'}, {'te...","[{'category': 'food', 'polarity': 'negative'}]","['Food', 'creme brulee', 'sugar']",Definition: The output will be the aspects (bo...,"['Food', 'creme brulee', 'sugar', 'kerosene']","[Food, creme brulee, sugar]"
...,...,...,...,...,...,...,...,...
2420,970,"Grilled whole fish wonderful, great spicing.","[{'term': 'fish', 'polarity': 'positive'}]","[{'category': 'food', 'polarity': 'positive'}]",['fish'],Definition: The output will be the aspects (bo...,"['Grilled whole fish', 'spicing']",[fish]
2421,2546,"Little Tonino's is just awesome, our favorite ...","[{'term': 'Gnochi', 'polarity': 'positive'}]","[{'category': 'food', 'polarity': 'positive'}]",['Gnochi'],Definition: The output will be the aspects (bo...,"['delivery place', 'Gnochi']",[Gnochi]
2422,2652,The restaurant is dark and not very attractive...,"[{'term': 'spot lights', 'polarity': 'negative'}]","[{'category': 'ambience', 'polarity': 'negativ...",['spot lights'],Definition: The output will be the aspects (bo...,"['spot lights', 'sunglasses']",[spot lights]
2423,1423,What makes this restaurant special are the aut...,"[{'term': 'sichuan cooking', 'polarity': 'posi...","[{'category': 'food', 'polarity': 'positive'},...","['sichuan cooking', 'chongqing hotpot']",Definition: The output will be the aspects (bo...,['chongqing hotpot'],"[sichuan cooking, chongqing hotpot]"


In [8]:
training_set = training_set[training_set['terms'].apply(lambda x: x != ['noaspectterm'])]

In [9]:
def replace_terms_with_blanks(row):
    pr_text = row['raw_text']
    for term in row['terms']:
        pr_text = pr_text.replace(term, '[MASK]')
    return pr_text
training_set['text_blank'] = training_set.apply(replace_terms_with_blanks, axis=1)

/tmp/ipykernel_1627144/3432187991.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  training_set['text_blank'] = training_set.apply(replace_terms_with_blanks, axis=1)


In [10]:
terms_lengths = training_set['terms'].apply(len).tolist()
mask_counts = training_set['text_blank'].str.count('\[MASK\]').tolist()
print(terms_lengths)
print(mask_counts)

# Find indices where the values in the two lists are different
different_indices = [i for i, (x, y) in enumerate(zip(terms_lengths, mask_counts)) if x != y]
print(len(different_indices))
training_set.drop(training_set.index[different_indices], inplace=True)

[2, 5, 4, 2, 3, 3, 1, 2, 2, 2, 1, 1, 1, 2, 1, 2, 3, 6, 2, 5, 1, 2, 1, 2, 1, 1, 3, 1, 2, 1, 2, 2, 4, 2, 2, 2, 2, 3, 1, 2, 5, 4, 3, 1, 2, 3, 4, 2, 2, 3, 1, 1, 1, 3, 4, 3, 1, 2, 1, 2, 3, 4, 1, 1, 2, 2, 1, 2, 1, 2, 2, 3, 2, 1, 1, 2, 3, 1, 3, 3, 3, 3, 1, 2, 1, 3, 4, 8, 1, 3, 1, 3, 2, 3, 2, 3, 3, 2, 9, 1, 2, 1, 3, 2, 2, 2, 1, 4, 3, 1, 1, 2, 1, 1, 3, 3, 2, 3, 4, 6, 3, 6, 4, 1, 2, 5, 3, 2, 3, 1, 3, 1, 4, 2, 3, 2, 1, 2, 2, 3, 3, 5, 3, 1, 3, 2, 4, 3, 1, 1, 3, 1, 2, 6, 2, 3, 2, 1, 1, 2, 3, 6, 3, 1, 3, 1, 6, 2, 1, 1, 3, 2, 3, 4, 3, 1, 2, 2, 3, 2, 1, 4, 2, 3, 2, 3, 1, 1, 3, 7, 1, 1, 3, 2, 5, 1, 2, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 2, 1, 2, 2, 1, 5, 1, 2, 2, 3, 1, 6, 3, 1, 2, 1, 3, 4, 2, 4, 2, 3, 3, 3, 2, 2, 7, 2, 1, 2, 1, 2, 1, 2, 3, 1, 6, 2, 2, 2, 1, 3, 1, 2, 1, 2, 4, 2, 2, 4, 6, 4, 2, 1, 2, 2, 2, 4, 3, 2, 3, 2, 1, 1, 2, 1, 1, 2, 3, 2, 2, 1, 1, 3, 2, 5, 3, 4, 4, 3, 2, 2, 3, 3, 1, 2, 2, 1, 3, 1, 4, 4, 2, 1, 1, 6, 1, 4, 2, 1, 2, 3, 3, 4, 2, 5, 1, 1, 1, 3, 1, 5, 6, 3, 3, 2, 2, 1, 3, 1, 3, 2, 1, 3, 

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_1627144/1207619775.py:2: SyntaxWarning: invalid escape sequence '\['
  mask_counts = training_set['text_blank'].str.count('\[MASK\]').tolist()
/tmp/ipykernel_1627144/1207619775.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  training_set.drop(training_set.index[different_indices], inplace=True)


In [11]:
import pickle
import random
with open('/home/s6moakba/Thesis/lex_terms_noun.pkl', 'rb') as f:
    lex_nouns = pickle.load(f)

In [12]:
multi_word_df = pd.read_csv('/home/s6moakba/Thesis/restaurant_multi_word_aspect_gpt.csv')
multi_word_set = set(multi_word_df['Aspect Term'])

In [13]:
weighted_combined_set = list(lex_nouns) + list(multi_word_set) * 6
random_item = random.sample(weighted_combined_set, 2)
print(random_item)

['humus', 'Bread and butter']


In [14]:
def add_random_terms(row, weigheted_combined_terms):
    terms = row['aspectTerms']
    new_terms = random.sample(weigheted_combined_terms, len(terms))
    return new_terms
training_set['new_terms'] = training_set.apply(lambda row: add_random_terms(row, weighted_combined_set), axis=1)

/tmp/ipykernel_1627144/4201400055.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  training_set['new_terms'] = training_set.apply(lambda row: add_random_terms(row, weighted_combined_set), axis=1)


In [16]:
training_set

,sentenceId,raw_text,aspectTerms,aspectCategories,labels,text,prediction,terms,text_blank,new_terms
0,2227,My suggestion is to eat family style because y...,"[{'term': 'Sushi roll (with avocado)', 'polari...","[{'category': 'food', 'polarity': 'neutral'}]","['dishes', 'eat family style']",Definition: The output will be the aspects (bo...,['dishes'],"[dishes, eat family style]",My suggestion is to [MASK] because you'll want...,"[Sushi roll (with avocado), griddle]"
1,2962,Three courses - choices include excellent muss...,"[{'term': 'Pork ribs', 'polarity': 'positive'}...","[{'category': 'food', 'polarity': 'positive'}]","['mussels', 'puff pastry goat cheese', 'sala...",Definition: The output will be the aspects (bo...,"['mussels', 'puff pastry goat cheese', 'sala...","[mussels, puff pastry goat cheese, salad with ...",Three [MASK] - choices include excellent [MASK...,"[Pork ribs, Pork and sauerkraut, water-colour,..."
2,2911,"Great food, good size menu, great service and ...","[{'term': 'drop', 'polarity': 'positive'}, {'t...","[{'category': 'food', 'polarity': 'positive'},...","['food', 'menu', 'service', 'setting']",Definition: The output will be the aspects (bo...,"['food', 'size menu', 'service', 'setting']","[food, menu, service, setting]","Great [MASK], good size [MASK], great [MASK] a...","[drop, biryani, Grilled salmon, Chicken and du..."
3,672,"Dip the ingredients in with your chopsticks, s...","[{'term': 'white', 'polarity': 'neutral'}, {'t...","[{'category': 'food', 'polarity': 'neutral'}]","['ingredients', 'chopsticks']",Definition: The output will be the aspects (bo...,['ingredients'],"[ingredients, chopsticks]","Dip the [MASK] in with your [MASK], swirl them...","[white, argyll]"
4,567,Food was average and creme brulee was awful - ...,"[{'term': 'cutting', 'polarity': 'neutral'}, {...","[{'category': 'food', 'polarity': 'negative'}]","['Food', 'creme brulee', 'sugar']",Definition: The output will be the aspects (bo...,"['Food', 'creme brulee', 'sugar', 'kerosene']","[Food, creme brulee, sugar]",[MASK] was average and [MASK] was awful - the ...,"[cutting, SALT, Chicken kebabs (with tzatziki)]"
...,...,...,...,...,...,...,...,...,...,...
2420,970,"Grilled whole fish wonderful, great spicing.","[{'term': 'Shrimp and grits', 'polarity': 'pos...","[{'category': 'food', 'polarity': 'positive'}]",['fish'],Definition: The output will be the aspects (bo...,"['Grilled whole fish', 'spicing']",[fish],"Grilled whole [MASK] wonderful, great spicing.",[Shrimp and grits]
2421,2546,"Little Tonino's is just awesome, our favorite ...","[{'term': 'communization', 'polarity': 'positi...","[{'category': 'food', 'polarity': 'positive'}]",['Gnochi'],Definition: The output will be the aspects (bo...,"['delivery place', 'Gnochi']",[Gnochi],"Little Tonino's is just awesome, our favorite ...",[communization]
2422,2652,The restaurant is dark and not very attractive...,"[{'term': 'ass', 'polarity': 'negative'}]","[{'category': 'ambience', 'polarity': 'negativ...",['spot lights'],Definition: The output will be the aspects (bo...,"['spot lights', 'sunglasses']",[spot lights],The restaurant is dark and not very attractive...,[ass]
2423,1423,What makes this restaurant special are the aut...,"[{'term': 'horseshoe', 'polarity': 'positive'}...","[{'category': 'food', 'polarity': 'positive'},...","['sichuan cooking', 'chongqing hotpot']",Definition: The output will be the aspects (bo...,['chongqing hotpot'],"[sichuan cooking, chongqing hotpot]",What makes this restaurant special are the aut...,"[horseshoe, cement]"


In [15]:
def replace_terms(row):
    aspect_terms = row['aspectTerms']
    generated_text = row['new_terms']
    for i, term in enumerate(aspect_terms):
        term['term'] = generated_text[i]
    return aspect_terms

training_set['aspectTerms'] = training_set.apply(replace_terms, axis=1)

/tmp/ipykernel_1627144/2893362978.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  training_set['aspectTerms'] = training_set.apply(replace_terms, axis=1)


In [17]:
def replace_mask(row):
    text = row['text_blank']
    for term in row['new_terms']:
        text = text.replace('[MASK]', term, 1)
    return text

training_set['text_blank'] = training_set.apply(replace_mask, axis=1)

/tmp/ipykernel_1627144/1593081476.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  training_set['text_blank'] = training_set.apply(replace_mask, axis=1)


In [34]:
roww = training_set.iloc[130]
print("raw_text:", roww['raw_text'])
print("original terms: ", roww['terms'])
print("new terms from dict: ",roww['new_terms'])

# print(roww['aspectTerms'])
print("new generated sentence:", roww['text_blank'])

raw_text: When he's not making authentic Neapolitan pizza in the open brick oven or lightly frying zucchini blossoms, he's visiting the regulars (a growing legion) and checking on newcomers.
original terms:  ['Neapolitan pizza', 'zucchini blossoms']
new terms from dict:  ['service', 'Mushroom risotto with truffle oil']
new generated sentence: When he's not making authentic service in the open brick oven or lightly frying Mushroom risotto with truffle oil, he's visiting the regulars (a growing legion) and checking on newcomers.


In [30]:
row = training_set.iloc[0]
print(row['aspectTerms'])
print(len(row['aspectTerms']))

[{'term': 'dishes', 'polarity': 'neutral'}, {'term': 'eat family style', 'polarity': 'positive'}]
2


# Check results


In [8]:
import pandas as pd 

df = pd.read_csv('/home/s6moakba/train_res_14_dynamic_sample_cv.csv')

In [9]:
df

,Unnamed: 0,blank,original_terms,original_text,prompt,aspectTerms_old
0,0,"Best of all is the warm [MASK], the [MASK] is ...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t..."
1,1,"Best of all is the warm [MASK], the [MASK] is ...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t..."
2,2,"Best of all is the warm [MASK], the [MASK] is ...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t..."
3,3,"Best of all is the warm [MASK], the [MASK] is ...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t..."
4,4,"Best of all is the warm [MASK], the [MASK] is ...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t..."
...,...,...,...,...,...,...
3937,3937,A wonderful [MASK]!,['place'],A wonderful place!,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'place', 'polarity': 'positive'}]"
3938,3938,A wonderful [MASK]!,['place'],A wonderful place!,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'place', 'polarity': 'positive'}]"
3939,3939,A wonderful [MASK]!,['place'],A wonderful place!,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'place', 'polarity': 'positive'}]"
3940,3940,A wonderful [MASK]!,['place'],A wonderful place!,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'place', 'polarity': 'positive'}]"
